# Phase 4 - Certified Power BI Exports

**Intended use:** Build certified CSV marts for Power BI (pages 1-5). Connect Power BI only to `data/exports/`, never bronze.

**Prerequisite:** Phases 1 and 3 (mart_readmission, predictions, metrics, champion register).


## 1. Setup


In [1]:
from __future__ import annotations
import json, hashlib, os, re, uuid, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    ROOT = Path("..").resolve()
os.chdir(ROOT)
DATAFILE = ROOT / "datafile.txt"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def load_registry(path=DATAFILE):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("|")
        if len(parts) < 4:
            continue
        rows.append({"role": parts[0].strip(), "zone": parts[1].strip(), "path": parts[2].strip(), "description": parts[3].strip()})
    return pd.DataFrame(rows)

def registry_paths(role=None):
    reg = load_registry()
    if role is not None:
        reg = reg[reg["role"] == role]
    return [ROOT / p for p in reg["path"].tolist()]

def registry_path(role, default=None):
    paths = registry_paths(role=role)
    if paths:
        return paths[0]
    return ROOT / default if default else None

def upsert_registry(role, zone, rel_path, description):
    lines = DATAFILE.read_text(encoding="utf-8").splitlines()
    new_line = f"{role}|{zone}|{rel_path}|{description}"
    out, found = [], False
    for line in lines:
        if line.startswith("#") or not line.strip():
            out.append(line)
            continue
        parts = line.split("|")
        if len(parts) >= 3 and parts[0].strip() == role and parts[2].strip() == rel_path:
            out.append(new_line)
            found = True
        else:
            out.append(line)
    if not found:
        out.append(new_line)
    DATAFILE.write_text("\n".join(out) + "\n", encoding="utf-8")

def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

def file_checksum(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("ROOT:", ROOT)
print(load_registry())


ROOT: E:\Amit\Project\Hospital project
           role    zone                                       path  \
0           raw  bronze                 data/raw/diabetic_data.csv   
1         clean  silver        data/lake/silver/encounters.parquet   
2      features    gold      data/lake/gold/model_features.parquet   
3     sequences    gold       data/lake/gold/rnn_sequences.parquet   
4   predictions    gold    data/lake/gold/test_predictions.parquet   
5       metrics    gold  data/lake/gold/experiment_results.parquet   
6          mart  export          data/exports/mart_readmission.csv   
7          mart  export        data/exports/mart_clinical_risk.csv   
8          mart  export    data/exports/mart_model_performance.csv   
9          mart  export         data/exports/mart_dq_scorecard.csv   
10     manifest     ops    data/lake/bronze/_manifests/latest.json   
11     champion     ops              models/champion_register.json   

                                      description 

## 2. Load certified inputs

Readmission mart, test predictions, experiment metrics, champion register.


In [2]:
exports = ROOT / "data" / "exports"
exports.mkdir(parents=True, exist_ok=True)

mart_readmission = pd.read_csv(exports / "mart_readmission.csv")
pred = pd.read_parquet(registry_path("predictions", "data/lake/gold/test_predictions.parquet"))
metrics = pd.read_parquet(registry_path("metrics", "data/lake/gold/experiment_results.parquet"))
register = json.loads((ROOT / "models" / "champion_register.json").read_text(encoding="utf-8"))
print("mart_readmission", mart_readmission.shape)
print("predictions", pred.shape)
print("metrics", metrics.shape)
print("champion", register.get("champion_model"))


mart_readmission (101766, 16)
predictions (7501, 13)
metrics (148, 13)
champion rf


## 3. Build `mart_clinical_risk`

Risk bands, high-risk alert flag, top-factor columns for Model Insights / Risk pages.


In [3]:
risk = pred.copy()
risk["alert_high_risk"] = (risk["risk_band"] == "High").astype(int)
top = register.get("top_features", [])
for i, f in enumerate(top[:5], 1):
    risk[f"top_factor_{i}"] = f.get("feature", "")
risk.to_csv(exports / "mart_clinical_risk.csv", index=False)
upsert_registry("mart", "export", "data/exports/mart_clinical_risk.csv", "Certified BI mart - risk scores and factors")
print("mart_clinical_risk", risk.shape, "high-risk rate", risk["alert_high_risk"].mean())


mart_clinical_risk (7501, 19) high-risk rate 0.013464871350486601


In [ ]:
# Shadow disagreement mart (optional BI storytelling)
import joblib
from inference.shadow import score_shadow, disagreement

shadow_path = ROOT / "models" / "shadow_tri_ensemble.joblib"
if shadow_path.exists() and len(pred):
    feat_cols = json.loads((ROOT / "models" / "champion_features.json").read_text(encoding="utf-8"))["features"]
    sample = pred.head(500).merge(mart_readmission[["encounter_id"] + [c for c in feat_cols if c in mart_readmission.columns]], on="encounter_id", how="left")
    rows = []
    for _, r in sample.iterrows():
        X = pd.DataFrame([{c: r.get(c, np.nan) for c in feat_cols}])
        champ_p = float(r.get("y_prob", np.nan))
        shadow_p = score_shadow(X)
        if shadow_p is None or np.isnan(champ_p):
            continue
        rows.append({
            "encounter_id": r["encounter_id"],
            "champion_prob": champ_p,
            "shadow_prob": shadow_p,
            "disagree": disagreement(champ_p, shadow_p),
        })
    if rows:
        shadow_mart = pd.DataFrame(rows)
        shadow_mart.to_csv(exports / "mart_shadow_disagreement.csv", index=False)
        print("mart_shadow_disagreement", shadow_mart.shape, "disagree rate", shadow_mart["disagree"].mean())

## 4. Build `mart_model_performance`

Full experiment matrix for page 5 slicers (model / split / horizon).


In [4]:
metrics.to_csv(exports / "mart_model_performance.csv", index=False)
upsert_registry("mart", "export", "data/exports/mart_model_performance.csv", "Certified BI mart - model performance")
print(metrics.groupby("model").size())


model
catboost             21
gb_ensemble          21
lightgbm             21
logreg               21
rf                   21
stacking_ensemble     1
tri_ensemble         21
xgboost              21
dtype: int64


## 5. Actual vs predicted series

Cumulative rates for the page-5 line chart.


In [5]:
avs = pred[["y_true", "y_pred", "y_prob"]].copy().reset_index(drop=True)
avs["idx"] = np.arange(len(avs))
avs["actual_cum_rate"] = avs["y_true"].expanding().mean()
avs["predicted_cum_rate"] = avs["y_pred"].expanding().mean()
avs.to_csv(exports / "mart_actual_vs_predicted.csv", index=False)
print(avs.head())


   y_true  y_pred    y_prob  idx  actual_cum_rate  predicted_cum_rate
0       0       0  0.405286    0              0.0            0.000000
1       0       1  0.473559    1              0.0            0.500000
2       0       1  0.584910    2              0.0            0.666667
3       0       0  0.290048    3              0.0            0.500000
4       0       1  0.583235    4              0.0            0.600000


## 6. KPI snapshot (metric dictionary)

Single JSON snapshot aligned with certified metric definitions.


In [6]:
kpi = {
    "total_patients": int(mart_readmission["patient_nbr"].nunique()),
    "total_encounters": int(len(mart_readmission)),
    "readmission_rate_30d": float(mart_readmission["readmit_30d"].mean()),
    "avg_los": float(mart_readmission["time_in_hospital"].mean()),
    "high_risk_rate": float((pred["risk_band"] == "High").mean()),
    "champion_model": register.get("champion_model"),
    "champion_recall": register.get("metrics", {}).get("recall"),
    "champion_roc_auc": register.get("metrics", {}).get("roc_auc"),
}
(exports / "kpi_snapshot.json").write_text(json.dumps(kpi, indent=2), encoding="utf-8")
print("KPIs", kpi)


KPIs

 {'total_patients': 71518, 'total_encounters': 101766, 'readmission_rate_30d': 0.11159915885462728, 'avg_los': 4.395986871843248, 'high_risk_rate': 0.013464871350486601, 'champion_model': 'rf', 'champion_recall': 0.5513126491646778, 'champion_roc_auc': 0.6496991006151235}


## 7. Build Power BI master CSV

Stack all certified exports into `powerbi_dashboard_master.csv` for single-import Power BI setup.


In [ ]:
import subprocess
import sys

builder = ROOT / "scripts" / "build_powerbi_master_csv.py"
result = subprocess.run([sys.executable, str(builder)], capture_output=True, text=True, cwd=ROOT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr or "build_powerbi_master_csv.py failed")

master_path = exports / "powerbi_dashboard_master.csv"
master = pd.read_csv(master_path, low_memory=False)
print(f"Master CSV: {master_path} — {master.shape[0]:,} rows × {master.shape[1]} columns")
print(master["record_type"].value_counts().to_string())


## 8. Power BI build instructions

How to wire the five dashboard pages in Power BI Desktop (see `powerbi/BUILD_INSTRUCTIONS.md`).


In [7]:
pbi_doc = ROOT / "powerbi" / "BUILD_INSTRUCTIONS.md"
if not pbi_doc.exists():
    raise FileNotFoundError(
        f"Missing {pbi_doc}. See docs/phase4_powerbi_dashboard.md for the certified export list."
    )
print("Power BI build instructions (source of truth):", pbi_doc)
print("Single import: data/exports/powerbi_dashboard_master.csv — see file for page mapping.")


Wrote E:\Amit\Project\Hospital project\powerbi\BUILD_INSTRUCTIONS.md


## 9. Phase 4 summary


In [8]:
print("Phase 4 exports complete")
files = sorted(p.name for p in exports.glob("*") if p.is_file())
print("Files in data/exports:", files)
assert (exports / "powerbi_dashboard_master.csv").exists(), "Missing master CSV — run section 7"


Phase 4 exports complete
Files in data/exports: ['eda', 'kpi_snapshot.json', 'mart_actual_vs_predicted.csv', 'mart_clinical_risk.csv', 'mart_dq_scorecard.csv', 'mart_model_performance.csv', 'mart_readmission.csv']
